### Imports

In [1]:
import torch
import torch.nn as nn
import copy
import random
import numpy as np
import torch.optim as optim
from tqdm.notebook import tqdm, trange
from torchvision import datasets, transforms
import time
import gc
from GPUtil import showUtilization as gpu_usage
from GPUtil import getGPUs
from torchvision.transforms import v2
import os, ctypes
from IPython.core.display import HTML
import os
import shutil
import json
from pathlib import Path

# For the correct functioning of tqdm
display(HTML("""
    <style>
        .jp-OutputArea-child:has(.jp-OutputArea-prompt:empty) {
              padding: 0 !important;
        }
    </style>
"""))

# To supress pesky warnings
import warnings
warnings.filterwarnings("ignore")

### Reproducibility

In [2]:
def set_seed(seed_value=42):
    np.random.seed(seed_value) # cpu vars
    random.seed(seed_value) # Python
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    os.environ['CUBLAS_WORKSPACE_CONFIG']=":4096:8"
    torch.manual_seed(seed_value) # cpu  vars
    torch.use_deterministic_algorithms(True,warn_only=True)
    torch.cuda.manual_seed_all(seed_value) # gpu vars
    torch.backends.cudnn.deterministic = True  #needed
    torch.backends.cudnn.benchmark = False

# 0. Useful functions

In this section we define all the functions we need, from creating and training the models to generating the datasets or cleaning unused memory.

### 0.1. Memory management

In [3]:
# Here we define a function to free all possible memory from the GPU to avoid memory issues when dealing with a lot of models
def free_gpu_cache(show_usage = False):
# This function is used to clear the GPU cache and avoid memory problems when dealing with large populations and big models
    if show_usage:
        print("Initial GPU Usage")
        gpu_usage()
    torch.cuda.ipc_collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    gc.collect()
    if show_usage:
        print("GPU Usage after emptying the cache")
        gpu_usage()

### 0.2. Model training functions

In [4]:
def trainloop(base_model, transforms, max_epochs = 256, lr = 1e-3, patience = 0, model_save_path = ""):
    try:
        accs = {"train" : [],
                "val" : [],
                "test" : []
               }
        times = []
        with tqdm(total = len(transforms), leave = False, desc='Data Augmentation scheme') as pbar0:
            for t_idx, t in enumerate(transforms):
                model = copy.deepcopy(base_model)
                
                if t_idx == len(transforms)-1: # For the no Data Aug. with reinitialized weights case
                    model.apply(weight_reset)
                
                start = time.time()
                val_loader, train_loader = retrieve_datasets(t)
        
                model.to(model.device)
                optimizer = optim.AdamW(model.parameters(), lr)
                
                model.train()
                training_loss = []
                validation_loss = []
                overfit = 0
                best_loss = np.inf
                with tqdm(total = max_epochs , leave = False, desc='Training model') as pbar1:
                    for epoch in range(max_epochs ):
                        running_loss = 0.0
                        with tqdm(total=len(train_loader), leave = False, desc='Epoch progress') as pbar2:
                            for inputs, labels in train_loader:
                                time_start = time.time()
                                inputs, labels = inputs.to(model.device), labels.to(model.device)
                                optimizer.zero_grad()
                                outputs = model(inputs).squeeze()
                                loss = criterion(outputs, labels)
                                #print(loss)
                                loss.backward()
                                optimizer.step()
                                running_loss += loss.item()
    
                                if time.time() - time_start > model.max_iter_time:
                                    pbar0.update(np.inf)
                                    pbar1.update(np.inf)
                                    pbar2.update(np.inf)
                                    del model, optimizer
                                    free_gpu_cache()
                                    raise Exception("This model would take too much time to train. Trying another random model.")
                                pbar2.update(1)
                        training_loss.append(running_loss/len(train_loader))
                        
                        model.eval()
                        running_loss = 0.
                        with tqdm(total=len(val_loader), leave = False, desc='Calculating validation loss') as pbar3:
                            with torch.no_grad():
                                for inputs, labels in val_loader:
                                    inputs, labels = inputs.to(model.device), labels.to(model.device)
                                    outputs = model.forward(inputs)
                                    loss = criterion(outputs, labels)
                                    running_loss += loss.detach().cpu().numpy()
                                    pbar3.update(1)
                        validation_loss.append(running_loss/len(val_loader))
                
                        if patience > 0:
                            if validation_loss[-1] < best_loss:
                                best_loss = validation_loss[-1]
                                overfit = 0
                                torch.save(model.state_dict(), "training_model.pt")
                            else:
                                overfit += 1
                            if overfit >= patience:
                                pbar1.update(max_epochs )
                                break
                        pbar1.update(1)
                    
                if patience > 0:
                    model.load_state_dict(torch.load("training_model.pt"))
            
                accs["train"] = accs["train"] + [accmetric(model, train_loader)]
                accs["val"] = accs["val"] + [accmetric(model, val_loader)]
                accs["test"] = accs["test"] + [accmetric(model, test_loader)]
                
                torch.save(model, "data_aug_"+str(t_idx)+".pt")
                
                del model, optimizer, loss, inputs, labels, outputs
                times += [time.time()-start]
                
                pbar0.update()
                free_gpu_cache()
        
        valid = np.sum([accs["val"][i]>0.2 for i in range(len(transforms))]) == len(transforms)
        if valid:
            for t_idx in range(len(transforms)):
                shutil.copyfile("data_aug_"+str(t_idx)+".pt", model_save_path+"data_aug_"+str(t_idx)+"/"+str(len(os.listdir(model_save_path+"data_aug_"+str(t_idx))))+".pt")
            return accs, times
        else:
            return None, None
    
    except Exception as e:
        pbar0.update(np.inf)
        pbar1.update(np.inf)
        pbar2.update(np.inf)
        print(e)
        del model, optimizer
        free_gpu_cache()
        raise Exception("This model would not fit in the GPU. Trying another random model.")

def accmetric(model, dataloader):
    model.to(model.device)
    model.eval()
    
    accuracy = 0
    # Turn off gradients for validation, saves memory and computations
    with tqdm(total=len(dataloader), leave = False, desc='Calculating accuracy') as pbar:
        with torch.no_grad():
            for inputs, labels in dataloader:
                inputs, labels = inputs.to(model.device), labels.to(model.device)
                logprobs = model.forward(inputs)
                top_p, top_class = logprobs.topk(1, dim=1)
                equals = (top_class == labels.view(inputs.shape[0], 1))
                accuracy += torch.mean(equals.type(torch.FloatTensor))
                pbar.update(1)
    model.to("cpu")
    free_gpu_cache()
    return accuracy.detach().numpy()/len(dataloader)


### 0.3. Model generation

In [5]:
class conv_block(nn.Module):
    def __init__(self, in_c, out_c, kernel_size = 3):
        self.kernel_size = kernel_size
        padding = int((kernel_size + 1)/2)
        self.padding = padding
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c, kernel_size=kernel_size, padding=padding)
        self.bn = nn.BatchNorm2d(out_c)
        self.relu = nn.ReLU()
        
    def forward(self, inputs):
        x = self.conv(inputs)
        x = self.bn(x)
        x = self.relu(x)
        return x  

class block(nn.Module):
    def __init__(self, in_c, out_c, resizing_type = None, from_block_channels = 0, kernel_size = 3):
        super().__init__()
        self.resizing_type = resizing_type
        self.kernel_size = kernel_size
        self.in_c = in_c
        self.out_c = out_c
        self.from_block_channels = from_block_channels
        
        if resizing_type == "D":
            self.resizing = nn.MaxPool2d((2, 2))
        elif resizing_type == "U":
            self.resizing = nn.ConvTranspose2d(in_c, in_c, kernel_size=2, stride=2, padding=0)
        else:
            self.resizing = nn.Identity()
        
        self.conv = conv_block(from_block_channels+in_c, out_c, kernel_size = kernel_size)
        
    def forward(self, inputs, skip):
        x = self.resizing(inputs)
        
        for extra_input in skip:
            x = torch.cat([x, nn.functional.interpolate(extra_input, size = [x.shape[2], x.shape[3]])], axis=1)
        x = self.conv(x)
        return x

class optimized_network(nn.Module):
    """
    A Convolutional Neural Network generated randomly.
    """
    def __init__(self, dataloader, n_blocks = 1, task = "infere", out_size = None, logmin_c = 5, logmax_c = 8, max_k = 9, complex_classifier = False, max_iter_time = 1, verbose = False):
        super().__init__()
        
        if hasattr(n_blocks, '__iter__'):
            if len(n_blocks) not in [1,2]:
                raise Exception("Invalid range for model initial layers.")
        elif n_blocks == int(n_blocks):
            n_blocks = [int(n_blocks), int(n_blocks)]
        else:
            raise Exception("Model initial layers needs to be either a scalar or a scalar range.")

        self.dataloader = dataloader
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.logmin_c = logmin_c
        self.logmax_c = logmax_c
        self.max_k = max_k
        self.complex_classifier = complex_classifier
        self.max_iter_time = max_iter_time
        
        batch = next(iter(dataloader))
        self.in_size = batch[0].shape
        self.label_size = batch[1].shape
        
        targets = dataloader.dataset.targets
        
        if task not in ["image_to_image", "image_to_mask", "object_detection", "regression", "classification", "infere"]:
            print("Unknown task. Infering task from dataset.")
            task = "infere"
        if task == "infere":
            if batch[0].shape[-2:] == batch[1].shape[-2:]:
                if batch[1].is_floating_point():
                    self.task = "image_to_image"
                else:
                    if batch[1].shape[-3] == 1:
                        self.task = "image_to_mask"
                    else:
                        self.task = "object_detection"
            else:
                if batch[1].is_floating_point():
                    self.task = "regression"
                else:
                    self.task = "classification"
        else:
            self.task = task

        if out_size:
            self.out_size = out_size
        else:
            if self.task == "classification" or self.task == "object_detection":
                self.out_size = max(targets) + 1
            else:
                self.out_size = batch[1].shape[1:]
        
        invalid = self.initialize_network(n_blocks = random.randint(n_blocks[0], n_blocks[1]), verbose = verbose)
        while invalid:
            del self.features, self.fc
            free_gpu_cache()
            invalid = self.initialize_network(n_blocks = random.randint(n_blocks[0], n_blocks[1]), verbose = verbose)

    def calculate_possible_connections(self):
        updown = [0]
        for l in self.features:
            if type(l.resizing) == nn.Identity:
                r = 0
            elif type(l.resizing) == nn.ConvTranspose2d:
                r = 1
            elif type(l.resizing) == nn.MaxPool2d:
                r = -1
            updown = updown + [updown[-1] + r]
        updown = updown[1:]
    
        possible_connections = []
        for idxin, lin in enumerate(updown):
            for idxout, lout in enumerate(updown[idxin+2:]):
                if lin == lout:
                    possible_connections += [(idxin, idxin+2+idxout)]
        
        return possible_connections
    
    def correct_blocks(self):
        for l, blk in enumerate(list(self.features.children())): # We check each block's new input size, rebuilding it if necessary
            if l == 0:
                if not blk.in_c == self.in_size[1]:
                    self.features[l] = block(in_c = self.in_size[1], out_c = blk.out_c, resizing_type = blk.resizing_type, from_block_channels = 0, kernel_size = blk.kernel_size)
    
            else:
                from_blocks = [i for i, x in enumerate(self.connections) if l in x]
                from_block_channels = 0
                for from_block in from_blocks:
                    from_block_channels += self.features[from_block].out_c
                
                if not (blk.in_c == self.features[l-1].out_c and blk.from_block_channels == from_block_channels):
                    self.features[l] = block(in_c = self.features[l-1].out_c, out_c = blk.out_c, resizing_type = blk.resizing_type, from_block_channels = from_block_channels, kernel_size = blk.kernel_size)
        try:
            self.build_fc()
        except:
            raise Exception("This model would not fit in the GPU. Trying another random model.")
    
    def initialize_network(self, n_blocks, verbose):
        try:
            self.connections = [[]] * n_blocks
            self.features = nn.Sequential()
            self.genome = []
            for l in range(n_blocks):
                if l == 0:
                    in_c = self.in_size[1]
                else:
                    in_c = out_c
                
                if l == n_blocks-1 and self.task != "classification":
                    out_c = self.out_size
                else:
                    out_c = 2**random.randint(self.logmin_c, self.logmax_c)
                
                resizing_type = random.choice(["U", "D", ""])
                
                kernel_size = random.randint(0,int((self.max_k-1)/2))*2+1
                self.features = nn.Sequential(*list(self.features.children()), block(in_c = in_c, out_c = out_c, resizing_type = resizing_type, from_block_channels = 0, kernel_size = kernel_size))

            possible_connections = self.calculate_possible_connections()
            k = random.randint(0, len(possible_connections))
            initial_connections = random.sample(possible_connections, k=k)

            for c in initial_connections:
                self.connections[c[0]] = self.connections[c[0]] + [c[1]]
            
            for i in range(len(self.connections)):
                self.connections[i].sort() # We order the connections so that they are tidy

            self.correct_blocks()
            return False
        except Exception as e:
            if verbose:
                print(e)
            return True

    def build_fc(self):
        self.fc = nn.Sequential(nn.Identity())
        if self.task == "classification":
            feat_size = np.prod([*self.to(self.device).forward(torch.zeros(self.in_size, device = self.device)).shape[1:]])
            free_gpu_cache()
            if self.complex_classifier:
                self.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(feat_size, self.out_size*4), nn.ReLU(), nn.Dropout(0.5), nn.Linear(self.out_size*4, self.out_size*2), nn.ReLU(), nn.Dropout(0.5), nn.Linear(self.out_size*2, self.out_size), nn.LogSoftmax(dim=0))
            else:
                self.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(feat_size, self.out_size), nn.LogSoftmax(dim=0))

    def forward(self, inputs):
        skip = [[]]*len(self.features)
        x = inputs
        for i, l in enumerate(self.features):
            x = l(x, skip[i])
            for to_block in self.connections[i]:
                skip[to_block] = skip[to_block]+[x]
        if (self.task == "classification" or self.task == "regression"):
            x = torch.flatten(x, start_dim=1)
            for l in self.fc:
                x = l(x)
        else:
            x = nn.functional.interpolate(x, size = [inputs.shape[2], inputs.shape[3]])
        del skip
        return x

def weight_reset(layer):
    if hasattr(layer, 'reset_parameters'):
        layer.reset_parameters()

### 0.5. Dataset retrieval with Data Augmentation

In [6]:
def retrieve_datasets(transforms, val_len = 5000, val_da = 10):
    trainset = datasets.CIFAR10(root='./data', train=True, download=[False if os.path.isfile(".\\data\\cifar-10-python.tar.gz") else True][0], transform=transforms)
    train_loader = torch.utils.data.DataLoader(trainset, batch_size=32, shuffle=True)
    # To compare the models fairly, we take val_da augmented copies of each validation image to create a fixed but augmented validation partition
    val_dset = []
    for _ in range(val_da):
        val_dset += [trainset[i] for i in range(val_len)]
    val_loader = torch.utils.data.DataLoader(val_dset, batch_size=32, shuffle=False)
    train_loader.dataset.data = train_loader.dataset.data[val_len:,:,:]
    train_loader.dataset.targets = train_loader.dataset.targets[val_len:]
    return val_loader, train_loader


# 1. Random model generation for CIFAR-10

In this section, models will be generated at random and trained with 3 Data Augmentation schemes besides to compare with a non-augmented training. The first one consists on a series of geometrical operations, the second one consists mainly on a gaussian filter with a very big filter (which is meant to degrade the images) and the third one utilizes PyTorch's AutoAugment policy for CIFAR-10.

### 1.1. Define the transforms with and without data augmentation

In [7]:
transforms_vanilla = v2.Compose([v2.ToTensor(),
                                 v2.ToDtype(torch.float32, scale=True),
                                 v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                                ])

transforms_dataug1 = v2.Compose([v2.ToTensor(),
                                 v2.RandomHorizontalFlip(p=0.5),
                                 v2.RandomAffine(degrees=30, scale=(0.9, 1.1), shear=10),
                                 v2.ToDtype(torch.float32, scale=True),
                                 v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                                ])

transforms_dataug2 = v2.Compose([v2.ToTensor(),
                                 v2.GaussianBlur(kernel_size=9, sigma=(0.5,1.5)),
                                 v2.RandomHorizontalFlip(p=0.5),
                                 v2.RandomEqualize(p = 0.5),
                                 v2.ToDtype(torch.float32, scale=True),
                                 v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                                ])

transforms_dataug3 = v2.Compose([v2.ToTensor(),
                                 v2.AutoAugment(v2.AutoAugmentPolicy.CIFAR10), # We use Pytorch's AutoAugment for CIFAR10
                                 v2.ToDtype(torch.float32, scale=True),
                                 v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                                ])

transforms = [transforms_vanilla, transforms_dataug1, transforms_dataug2, transforms_dataug3, transforms_vanilla] # The second transforms_vanilla is for the reinitialized weights case

# Download and load the test data, with no augmentations
testset = datasets.CIFAR10(root='./data', train=False, download=[False if os.path.isfile(".\\data\\cifar-10-python.tar.gz") else True][0], transform=transforms_vanilla)
test_loader = torch.utils.data.DataLoader(testset, batch_size=32, shuffle=False)

criterion = nn.NLLLoss()

Files already downloaded and verified


### 1.2. Run the model creation and testing

In [ ]:
# Model and training parameters
n_tests = 64
max_layers = 12
max_epochs = 256
patience = 4

model_args = {"dataloader" : test_loader, # To check the input and output sizes, type of problem, etc, from
              "task" : "infere",
              "out_size" : None,
              "logmin_c" : 4, # Number of channels per layer goes from 16 to 1024
              "logmax_c" : 10,
              "max_k" : 7, # Kernel size is at most 9 px wide
              "max_iter_time" : 2, # Only train models that require at most 2 seconds per batch
              "verbose" : True}

# Create folders to save models at
os.makedirs("saved_models/", exist_ok = True)

model_save_path = "saved_models/cifar10/"
os.makedirs(model_save_path, exist_ok = True)
for t in range(len(transforms)):
    os.makedirs(model_save_path+"data_aug_"+str(t), exist_ok = True)

tr_accs = {"vanilla" : [],"daug1" : [],"daug2" : [],"daug3" : [], "reinit" : []}
val_accs = {"vanilla" : [],"daug1" : [],"daug2" : [],"daug3" : [], "reinit" : []}
test_accs = {"vanilla" : [],"daug1" : [],"daug2" : [],"daug3" : [], "reinit" : []}
train_times = {"vanilla" : [],"daug1" : [],"daug2" : [],"daug3" : [], "reinit" : []}

seed_value = 0
with tqdm(total=n_tests, leave = False, desc='Generating and training models') as pbar_tests:
    while len(val_accs["vanilla"]) < n_tests:
        seed_value += 1
        try:
            free_gpu_cache()
            set_seed(seed_value)
            base_model = optimized_network(**model_args, n_blocks = random.randint(1, max_layers), complex_classifier = random.randint(0, 1))
            accs, times = trainloop(base_model, transforms, max_epochs = max_epochs, lr = 1e-3, patience = patience, model_save_path = model_save_path)

            if not accs == None:
                tr_accs["vanilla"] = tr_accs["vanilla"] + [accs["train"][0]]
                tr_accs["daug1"] = tr_accs["daug1"] + [accs["train"][1]]
                tr_accs["daug2"] = tr_accs["daug2"] + [accs["train"][2]]
                tr_accs["daug3"] = tr_accs["daug3"] + [accs["train"][3]]
                tr_accs["reinit"] = tr_accs["reinit"] + [accs["train"][4]]
                
                val_accs["vanilla"] = val_accs["vanilla"] + [accs["val"][0]]
                val_accs["daug1"] = val_accs["daug1"] + [accs["val"][1]]
                val_accs["daug2"] = val_accs["daug2"] + [accs["val"][2]]
                val_accs["daug3"] = val_accs["daug3"] + [accs["val"][3]]
                val_accs["reinit"] = val_accs["reinit"] + [accs["val"][4]]
                
                test_accs["vanilla"] = test_accs["vanilla"] + [accs["test"][0]]
                test_accs["daug1"] = test_accs["daug1"] + [accs["test"][1]]
                test_accs["daug2"] = test_accs["daug2"] + [accs["test"][2]]
                test_accs["daug3"] = test_accs["daug3"] + [accs["test"][3]]
                test_accs["reinit"] = test_accs["reinit"] + [accs["test"][4]]
    
                train_times["vanilla"] = train_times["vanilla"] + [times[0]]
                train_times["daug1"] = train_times["daug1"] + [times[1]]
                train_times["daug2"] = train_times["daug2"] + [times[2]]
                train_times["daug3"] = train_times["daug3"] + [times[3]]
                train_times["reinit"] = train_times["reinit"] + [times[4]]

                Path("tr_accs.json").write_text(json.dumps(tr_accs))
                Path("val_accs.json").write_text(json.dumps(val_accs))
                Path("test_accs.json").write_text(json.dumps(test_accs))
                Path("train_times.json").write_text(json.dumps(train_times))

                pbar_tests.update()
            else:
                print("Generated useless model, retrying")
                
        except Exception as e:
            print(e)


Generating and training models:   0%|          | 0/64 [00:00<?, ?it/s]

Data Augmentation scheme:   0%|          | 0/5 [00:00<?, ?it/s]

Files already downloaded and verified


Training model:   0%|          | 0/256 [00:00<?, ?it/s]

Epoch progress:   0%|          | 0/1407 [00:00<?, ?it/s]

...

Calculating validation loss:   0%|          | 0/1563 [00:00<?, ?it/s]

Epoch progress:   0%|          | 0/1407 [00:00<?, ?it/s]

Calculating validation loss:   0%|          | 0/1563 [00:00<?, ?it/s]

Epoch progress:   0%|          | 0/1407 [00:00<?, ?it/s]

Calculating validation loss:   0%|          | 0/1563 [00:00<?, ?it/s]

### 1.3. Resulting accuracies and training times

In [3]:
print("tr_accs =", tr_accs)
print("val_accs =", val_accs)
print("test_accs =", test_accs)
print("train_times =", train_times)

tr_accs = {'vanilla': [None, 0.7036691542288557, 0.6876554726368159, 0.8362873134328358, None, None, 0.7523987206823027, None, None, 0.773476368159204, None, None, None, 0.8147876687988628, 0.7348525230987918, None, None, 0.8483475479744137, 0.8896366382373845, 0.7721215351812367, None, 0.7883795309168443, 0.7858919687277897, None, 0.8202292110874201, 0.9678837953091685, 0.8154095593461265, 0.6314410092395167, 0.8294020966595593, None, None, 0.7236807036247335, 0.8080801350390903, None, None, 0.8922574626865671, 0.8121002132196162, 0.8546330845771144, None, 0.8608519900497512, None, 0.7447805614783227, None, 0.8060589907604833, None, None, 0.7665467306325515, None, None, None, 0.9032960199004975, 0.8186078535891969, 0.7716329068941009, None, 0.7123756218905473, None, 0.9484052949538024, 0.8543665600568585, None, 0.8439943141435678, 0.7723880597014925, None, 0.7961087420042644, None, 0.8305126154939588, None, 0.9311922530206113, None, 0.7892235252309879, 0.7615049751243781, 0.7758306680